In [2]:
import pandas as pd
import time
import os
from collections import deque
import urllib.request, urllib.error, urllib.parse
import json
from dotenv import load_dotenv

REST_URL = "http://data.bioontology.org"

# Create a .env file with your API key
# Format: BIO_PORTAL_API_KEY=your_api_key_here
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")


# --- CONFIGURATION ---
# Output file name
OUTPUT_FILE = "doid_disease_tree.csv"

# Batch size for saving progress (save every N nodes processed)
SAVE_INTERVAL = 50 

# Rate limiting (seconds between API calls)
API_DELAY = 0.1 

def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())


In [3]:
# Access the DOID ontology specifically
doid_acronym = "DOID"  # Human Disease Ontology
doid_url = f"{REST_URL}/ontologies/{doid_acronym}"
doid_ontology = get_json(doid_url)
print(f"Accessing DOID: {doid_ontology['name']}")

# Get the root classes of the DOID ontology
roots_url = doid_ontology['links']['roots']
roots = get_json(roots_url)
print("\nRoot classes in DOID:")
if len(roots) > 1:
    for root in roots:
        if root['prefLabel'] == 'disease':
            disease_root = root
            print("\tFOUND ROOT:\n", root)
        

Accessing DOID: Human Disease Ontology

Root classes in DOID:
	FOUND ROOT:
 {'prefLabel': 'disease', 'synonym': [], 'definition': ['A disease is a disposition (i) to undergo pathological processes that (ii) exists in an organism because of one or more disorders in that organism.'], 'cui': [], 'semanticType': [], 'obsolete': False, 'created': None, 'modified': None, 'memberOf': [], 'inScheme': [], '@id': 'http://purl.obolibrary.org/obo/DOID_4', '@type': 'http://www.w3.org/2002/07/owl#Class', 'links': {'self': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4', 'ontology': 'https://data.bioontology.org/ontologies/DOID', 'children': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/children', 'parents': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/parents', 'descendants': 'https://data.bioontology.org/ontologies/DOID/classes/http%

In [4]:
disease_root

{'prefLabel': 'disease',
 'synonym': [],
 'definition': ['A disease is a disposition (i) to undergo pathological processes that (ii) exists in an organism because of one or more disorders in that organism.'],
 'cui': [],
 'semanticType': [],
 'obsolete': False,
 'created': None,
 'modified': None,
 'memberOf': [],
 'inScheme': [],
 '@id': 'http://purl.obolibrary.org/obo/DOID_4',
 '@type': 'http://www.w3.org/2002/07/owl#Class',
 'links': {'self': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4',
  'ontology': 'https://data.bioontology.org/ontologies/DOID',
  'children': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/children',
  'parents': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/parents',
  'descendants': 'https://data.bioontology.org/ontologies/DOID/classes/http%3A%2F%2Fpurl.obolibrary.org%2Fobo%2FDOID_4/descendants',

# Load CSV generated from bfs_disease

In [12]:
import pandas as pd

df = pd.read_csv("doid_disease_tree.csv")
print("Shape: ", df.shape)
df.head(3)

Shape:  (11421, 4)


,id,level,path,prefLabel
0,http://purl.obolibrary.org/obo/DOID_4,0,disease,disease
1,http://purl.obolibrary.org/obo/DOID_0060501,1,disease/metal allergy,metal allergy
2,http://purl.obolibrary.org/obo/DOID_1417,1,disease/choroid disease,choroid disease


In [14]:
# Count the total number of unique prefLabel
df.prefLabel.nunique()


11421

In [17]:
d = "pemphigus foliaceus"
d_list = df.prefLabel.tolist()

d  in d_list

True

In [18]:
d2 = "contractures, pterygia, and spondylocarpotarsal fusion syndrome 1B"
d2  in d_list

True